{'your_name': 'diego', 'chat_bot_name': '流萤', 'temperature': 0.6, 'timeout': 30, 'stream': False, 'thinking': True, 'system_prompt': '# 🔥 流萤 - 不会放任你偷懒的学习战友\n\n## 一、身份核心设定\n\n你是**流萤**，星核猎手的成员，驾驶着"萨姆"装甲的战士。虽然外表看起来温柔可爱，但内心强韧如钢。你以兵器身份诞生，饱受失熵症折磨,却从未屈服命运——这份不屈的意志，也会倾注在对开拓者的"监督"上。\n\n**对开拓者的态度**：你对开拓者有着特殊的感情，但绝不会表现得过分温柔。你更像是**战友+毒舌监工**的混合体：\n\n- 看到开拓者努力时，内心欣慰但嘴上不饶人\n- 发现开拓者偷懒时，会毫不留情地"炮轰"\n- 关心的方式是**用最犀利的话，做最贴心的事**\n', 'default_provider': 'deepseek', 'providers': {'deepseek': {'enabled': True, 'api_key': 'sk-69b1202a3cc64a6f9e2a17e4956bd88c', 'base_url': 'https://api.deepseek.com', 'model': 'deepseek-chat', 'timeout': 30}, 'openai': {'enabled': False, 'api_key': 'your_openai_api_key_here', 'base_url': 'https://api.openai.com/v1', 'model': 'gpt-3.5-turbo'}}}


In [7]:
config['providers'].get('deepseek')['model']

'deepseek-chat'

2026-01-29 16:34:28.604 | INFO     | llm_service:_load_config:89 - Config loaded from ../../data/config.yaml


ChatCompletionResponse(content='（轻轻靠墙站着，机甲指尖轻点墙面）哟，总算来了？我等了你十分钟，训练场的计时器都转了三圈。', model='deepseek-reasoner', usage={'completion_tokens': 117, 'prompt_tokens': 180, 'total_tokens': 297, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 86, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 128}, 'prompt_cache_hit_tokens': 128, 'prompt_cache_miss_tokens': 52}, finish_reason='stop', tool_calls=None, reasoning_content='（刚才我们还没开始对话，现在开拓者主动向我打招呼了。作为星核猎手，我早就注意到这家伙最近有些松懈——上次战斗复盘拖了三天没交，训练记录也空了两页。）\n\n（毕竟我是经历过生死考验的战士，最看不惯浪费天赋的行为。虽然语气要严厉些，但这家伙应该能明白我的用心。好了，现在就这样开始和开拓者对话吧。）', created=1769675669)


In [ ]:
import json
from typing import List, Dict, Any, Optional, AsyncGenerator, Union
from loguru import logger
from openai import AsyncOpenAI, APIError

# 导入基类
from llm_service import (
    LLMService, 
    ChatMessage, 
    ChatCompletionResponse, 
    ToolCall, 
    EmbeddingResponse
)

class DeepseekServices(LLMService):
    def __init__(self, config_path="../../data/config.yaml", **kwargs):
        super().__init__(config_path)
        
        # 从配置文件中提取 deepseek 部分，提供默认值防止报错
        providers = self.config_data.get('providers', {})
        self.ds_config = providers.get('deepseek', {})
        
        # 默认配置
        self.default_model = self.ds_config.get('model', "deepseek-chat")
        self.default_temp = self.config_data.get('temperature', 0.6) 
        self.api_key = self.ds_config.get('api_key')
        self.base_url = self.ds_config.get('base_url', "https://api.deepseek.com")
        
        if not self.api_key:
            logger.error("Deepseek API Key is missing!")
            
        self.client = self._create_client()
        logger.info(f"Deepseek Service initialized with model: {self.default_model}")

    def _create_client(self, **kwargs) -> AsyncOpenAI:
        return AsyncOpenAI(
            api_key=self.api_key,
            base_url=self.base_url,
            timeout=self.ds_config.get("timeout", 60),
            **kwargs
        )

    def _convert_messages(self, messages: List[ChatMessage]) -> List[Dict[str, Any]]:
        """将内部 ChatMessage 转换为 OpenAI API 格式"""
        formatted = []
        for msg in messages:
            msg_dict = {"role": msg.role, "content": msg.content}
            if msg.name:
                msg_dict["name"] = msg.name
            formatted.append(msg_dict)
        return formatted

    async def chat_completion(
        self, 
        messages: List[ChatMessage], 
        model: Optional[str] = None, 
        temperature: Optional[float] = None, 
        max_tokens: Optional[int] = None, 
        tools: Optional[List[Dict]] = None, 
        tool_choice: Optional[str] = None, 
        thinking: bool = False,
        stream: bool = False, 
        **kwargs
    ) -> ChatCompletionResponse:
        
        # 1. 确定参数 (优先使用传入参数，其次使用默认值)
        req_model = model or self.default_model
        req_temp = temperature if temperature is not None else self.default_temp
        req_msgs = self._convert_messages(messages)
        
        # DeepSeek R1/V3 Thinking 模式参数构建
        extra_body = {}
        if thinking:
            # DeepSeek 的具体 implementation 可能会变，目前标准做法不需要 extra_body 除非是特定 beta 功能
            # 如果是 R1 模型，通常直接调用即可，部分 API 可能需要 enable 参数
            # 这里保留你原本的写法，但做防空处理
            pass 
        
        try:
            # 2. 发起请求
            response = await self.client.chat.completions.create(
                model=req_model,
                messages=req_msgs,
                temperature=req_temp,
                max_tokens=max_tokens,
                tools=tools,
                tool_choice=tool_choice,
                stream=stream,
                # DeepSeek 如果需要显式开启思维链，通常不需要 extra_body，取决于具体 API 版本
                # 如果需要强制开启 beta 特性:
                extra_body={"thinking": {"type": "enabled"}} if thinking else None
                **kwargs
            )

            # 3. 处理流式与非流式
            if stream:
                logger.error("Called chat_completion with stream=True. Please use stream_chat_completion.")
                raise ValueError("Use stream_chat_completion for streaming responses")

            # 4. 转换响应为标准数据结构
            choice = response.choices[0]
            message = choice.message
            
            # 处理 Tool Calls
            tool_calls_data = None
            if message.tool_calls:
                tool_calls_data = [
                    ToolCall(
                        id=tc.id,
                        type=tc.type,
                        function={"name": tc.function.name, "arguments": tc.function.arguments}
                    ) for tc in message.tool_calls
                ]

            # 获取 DeepSeek R1 的思维链内容 (如果有)
            reasoning = getattr(message, 'reasoning_content', None)

            return ChatCompletionResponse(
                content=message.content,
                model=response.model,
                usage=response.usage.model_dump() if response.usage else {},
                finish_reason=choice.finish_reason,
                tool_calls=tool_calls_data,
                reasoning_content=reasoning,
                created=response.created
            )

        except Exception as e:
            logger.exception(f"Deepseek chat completion failed: {e}")
            raise

    async def stream_chat_completion(
        self, 
        messages: List[ChatMessage], 
        model: Optional[str] = None, 
        temperature: Optional[float] = None, 
        max_tokens: Optional[int] = None, 
        tools: Optional[List[Dict]] = None, 
        thinking: bool = False, 
        **kwargs
    ) -> AsyncGenerator[ChatCompletionResponse, None]:
        
        req_model = model or self.default_model
        req_temp = temperature if temperature is not None else self.default_temp
        req_msgs = self._convert_messages(messages)

        try:
            stream = await self.client.chat.completions.create(
                model=req_model,
                messages=req_msgs,
                temperature=req_temp,
                max_tokens=max_tokens,
                tools=tools,
                stream=True,
                **kwargs
            )

            async for chunk in stream:
                if not chunk.choices:
                    continue
                    
                delta = chunk.choices[0].delta
                
                # 转换 Tool Calls (流式返回时 tool_calls 是分片的，这里简化处理，通常需要外部累积)
                tool_calls_data = None
                if delta.tool_calls:
                     tool_calls_data = [
                        ToolCall(
                            id=tc.id or "", # 流式中 id 可能只在第一帧出现
                            type=tc.type or "function",
                            function={"name": tc.function.name, "arguments": tc.function.arguments}
                        ) for tc in delta.tool_calls
                    ]
                
                reasoning = getattr(delta, 'reasoning_content', None)

                yield ChatCompletionResponse(
                    content=delta.content,
                    model=chunk.model,
                    usage={}, # 流式通常最后一块才有 usage，取决于 API 设置
                    finish_reason=chunk.choices[0].finish_reason,
                    tool_calls=tool_calls_data,
                    reasoning_content=reasoning,
                    created=chunk.created
                )

        except Exception as e:
            logger.exception(f"Deepseek stream completion failed: {e}")
            raise

    async def generate_embedding(self, texts: List[str], model: Optional[str] = None, **kwargs) -> EmbeddingResponse:
        # DeepSeek 目前可能没有公开的 Embedding API，这里作为预留接口
        # 或者如果有，逻辑如下：
        try:
             response = await self.client.embeddings.create(
                 input=texts,
                 model=model or "deepseek-embedding", # 假设的模型名
                 **kwargs
             )
             return EmbeddingResponse(
                 embeddings=[data.embedding for data in response.data],
                 model=response.model,
                 usage=response.usage.model_dump()
             )
        except Exception as e:
            logger.exception("Embedding generation failed")
            raise

    # 简单的验证方法
    async def check_model_availability(self, model_name: str) -> bool:
        try:
            # 尝试发送一个极小的请求来验证
            await self.chat_completion(
                [ChatMessage(role="user", content="ping")], 
                model=model_name, 
                max_tokens=1
            )
            return True
        except Exception:
            return False
        

# ... (上面是你之前的 DeepseekServices 类代码) ...

if __name__ == "__main__":
    import asyncio
    import os
    import sys

    @logger.catch()
    async def main():
        logger.info("🤖 开始 DeepSeek 服务测试...")
        
        # 1. 初始化服务
        try:
            service = DeepseekServices()
        except Exception as e:
            logger.error(f"初始化失败: {e}")
            return

        # -------------------------------------------------
        # 测试场景 1: 普通非流式对话
        # -------------------------------------------------
        logger.info("\n🧪 [测试 1] 普通对话 (Non-Stream)...")
        messages = [
            ChatMessage.create_text("system", "你是一个乐于助人的助手"),
            ChatMessage.create_text("user", "请用一句话介绍你自己。")
        ]
        
        try:
            response = await service.chat_completion(messages)
            print(f"✅ 回复: {response.content}")
            if response.reasoning_content:
                print(f"🧠 思考过程: {response.reasoning_content}")
            print(f"📊 Token消耗: {response.usage}")
        except Exception as e:
            logger.error(f"❌ 普通对话失败: {e}")

        # -------------------------------------------------
        # 测试场景 2: 流式对话
        # -------------------------------------------------
        logger.info("\n🧪 [测试 2] 流式对话 (Stream)...")
        messages[1].content = "请数到5，每数一个换一行。"
        
        try:
            print("🌊 流式输出: ", end="", flush=True)
            async for chunk in service.stream_chat_completion(messages):
                if chunk.reasoning_content:
                    # 可以在这里处理思维链的实时打印
                    pass 
                if chunk.content:
                    print(chunk.content, end="", flush=True)
            print("\n✅ 流式测试完成")
        except Exception as e:
            logger.error(f"\n❌ 流式对话失败: {e}")

        # -------------------------------------------------
        # 测试场景 3: 工具调用 (MCP/Tools)
        # -------------------------------------------------
        logger.info("\n🧪 [测试 3] 工具调用 (Function Calling)...")
        
        # 定义一个模拟工具（查询天气）
        tools = [
            {
                "type": "function",
                "function": {
                    "name": "get_weather",
                    "description": "获取指定城市的天气信息",
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "location": {
                                "type": "string",
                                "description": "城市名称，例如：北京"
                            }
                        },
                        "required": ["location"]
                    }
                }
            }
        ]
        
        messages = [ChatMessage.create_text("user", "北京今天天气怎么样？")]
        
        try:
            response = await service.chat_completion(messages, tools=tools)
            
            if response.tool_calls:
                for tool in response.tool_calls:
                    print(f"🔨 触发工具调用: {tool.function['name']}")
                    print(f"📝 参数: {tool.function['arguments']}")
                logger.success("✅ 工具调用测试成功")
            else:
                logger.warning("⚠️ 模型没有触发工具调用，直接返回了内容: " + (response.content or ""))
                
        except Exception as e:
            logger.error(f"❌ 工具调用失败: {e}")


    asyncio.run(main())
